In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer

# import ensemble methods
from sklearn.ensemble import (
    BaggingClassifier,
    GradientBoostingClassifier
)
from xgboost import XGBClassifier

# import base estimators
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings(
    "ignore", category=DeprecationWarning
)  # to avoid deprecation warnings

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=[
        "#4B9AC7",
        "#4BE8E0",
        "#9DD4F3",
        "#97FBF6",
        "#2A7FAF",
        "#23B1AB",
        "#0E3449",
        "#015955",
    ]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg"  # to be replaced by "iframe" if working on JULIE

In [10]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("Data.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [11]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include="all")
display(data_desc)
print()

print("Percentage of missing values: ")
display(100 * dataset.isnull().sum() / dataset.shape[0])

Number of rows : 10

Display of dataset: 


,Country,Age,Salary,Purchased
0,France,44.0,72000,No
1,Spain,27.0,48000,Yes
2,Germany,30.0,54000,No
3,Spain,38.0,61000,No
4,Germany,40.0,69000,Yes



Basics statistics: 


,Country,Age,Salary,Purchased
count,10,9.000000,10.000000,10
unique,3,NaN,NaN,2
top,France,NaN,NaN,No
freq,4,NaN,NaN,5
mean,NaN,38.777778,64300.000000,NaN
std,NaN,7.693793,11681.419244,NaN
min,NaN,27.000000,48000.000000,NaN
25%,NaN,35.000000,55000.000000,NaN
50%,NaN,38.000000,64000.000000,NaN
75%,NaN,44.000000,71250.000000,NaN



Percentage of missing values: 


Country       0.0
Age          10.0
Salary        0.0
Purchased     0.0
dtype: float64

## Preprocessing


In [12]:
# Separate target variable Y from features X
print("Separating labels from features...")
target_variable = "Purchased"

X = dataset.drop(target_variable, axis=1)
Y = dataset.loc[:, target_variable]

print("...Done.")
print()

print("Y : ")
print(Y.head())
print()
print("X :")
print(X.head())

Separating labels from features...
...Done.

Y : 
0     No
1    Yes
2     No
3     No
4    Yes
Name: Purchased, dtype: str

X :
   Country   Age  Salary
0   France  44.0   72000
1    Spain  27.0   48000
2  Germany  30.0   54000
3    Spain  38.0   61000
4  Germany  40.0   69000


In [13]:
# Automatically detect names of numeric/categorical columns
numeric_features = []
categorical_features = []
for i, t in X.dtypes.items():
    if ("float" in str(t)) or ("int" in str(t)):
        numeric_features.append(i)
    else:
        categorical_features.append(i)

print("Found numeric features ", numeric_features)
print("Found categorical features ", categorical_features)

Found numeric features  ['Age', 'Salary']
Found categorical features  ['Country']


In [14]:
# Divide dataset Train set & Test set
print("Dividing into train and test sets...")
# WARNING : don't forget stratify=Y for classification problems
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=0, stratify=Y
)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [15]:
# Create pipeline for numeric features
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="mean"),
        ),  # missing values will be replaced by columns' mean
        ("scaler", StandardScaler()),
    ]
)

In [16]:
# Create pipeline for categorical features
categorical_transformer = OneHotEncoder(
    drop="first"
)  # no missing values in categorical data, so we only need the OHE

In [17]:
# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

In [ ]:
# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train = preprocessor.fit_transform(X_train)
print("...Done.")
print(
    X_train[0:5]
)  # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Label encoding
print("Encoding labels...")
print(Y_train.head())
encoder = LabelEncoder()
Y_train = encoder.fit_transform(Y_train)
print("...Done")
print(Y_train[0:5])
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(
    X_test
)  # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print("...Done.")

print(
    X_test[0:5, :]
)  # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

# Label encoding
print("Encoding labels...")
print(Y_test[0:5])
Y_test = encoder.transform(Y_test)
print("...Done")
print(Y_test[0:5])

Performing preprocessings on train set...
   Country   Age  Salary
0   France  44.0   72000
4  Germany  40.0   69000
6    Spain   NaN   52000
9   France  37.0   67000
3    Spain  38.0   61000
...Done.
[[ 1.61706195e+00  1.46885753e+00  0.00000000e+00  0.00000000e+00]
 [ 8.22715727e-01  1.09777773e+00  1.00000000e+00  0.00000000e+00]
 [-1.41104234e-15 -1.00500778e+00  0.00000000e+00  1.00000000e+00]
 [ 2.26956063e-01  8.50391200e-01  0.00000000e+00  0.00000000e+00]
 [ 4.25542617e-01  1.08231607e-01  0.00000000e+00  1.00000000e+00]]

Encoding labels...
0     No
4    Yes
6     No
9    Yes
3     No
Name: Purchased, dtype: str
...Done
[0 1 0 1 0]
Performing preprocessings on test set...
   Country   Age  Salary
8  Germany  50.0   83000
7   France  48.0   79000
...Done.
[[2.80858127 2.82948345 1.         0.        ]
 [2.41140816 2.33471038 0.         0.        ]]

Encoding labels...
8     No
7    Yes
Name: Purchased, dtype: str
...Done
[0 1]


## Bagging

### Bagging with logistic regression as base estimator


BaggingClassifier est un ensemble method qui va entrainer plusieurs LogisticRegression sur des échantillons différents de notre dataset (bootstrap sampling) et faire une moyenne de leurs prédictions pour faire la prédiction finale du modèle.


On effectue une recherche des meilleurs hyperparamètres pour un BaggingClassifier basé sur une régression logistique. La grille teste différents niveaux de régularisation (C) et le nombre de modèles dans l’ensemble (n_estimators). La validation croisée à 3 plis permet d’évaluer chaque combinaison, puis on affiche les meilleurs paramètres et les précisions sur les ensembles d’entraînement et de test.

In [ ]:
# Perform grid search
print("Grid search...")
logistic_regression = LogisticRegression(
    max_iter=1000
)  # max_iter changed because of convergence warning
model = BaggingClassifier(estimator=logistic_regression) 

# Grid of values to be tested
params = {
    "estimator__C": [ 
        0.01,
        0.05,
        0.1,
        0.5,
    ],  # base_estimator__ prefix because C is a parameter from LogisticRegression!
    
    "n_estimators": [
        5,
        10,
        20,
        30,
    ],  # n_estimators is a hyperparameter of the ensemble method
}
print(params)
gridsearch = GridSearchCV(
    model, param_grid=params, cv=3
)  # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__C': [0.01, 0.05, 0.1, 0.5], 'n_estimators': [5, 10, 20, 30]}
...Done.
Best hyperparameters :  {'estimator__C': 0.01, 'n_estimators': 10}
Best validation accuracy :  0.5

Accuracy on training set :  0.5
Accuracy on test set :  0.5


La recherche a trouvé que les meilleurs hyperparamètres pour le BaggingClassifier avec régression logistique sont `C = 0.01` et `n_estimators = 10`.

Cependant, la précision reste faible (0.5) sur la validation, l’entraînement et le test. Cela indique que le modèle **n’apprend pas bien les données**, probablement parce que la régression logistique est trop simple pour ce problème ou que les données ne sont pas linéairement séparables. Il faudrait envisager :

* d’utiliser un modèle plus complexe (par exemple un arbre de décision ou un réseau),
* ou de revoir le prétraitement et les features.


### Bagging with decision tree as base estimator


On recherche les meilleurs hyperparamètres pour un BaggingClassifier basé sur un arbre de décision. La grille teste la profondeur de l’arbre, le nombre minimum d’échantillons par feuille et par split, ainsi que le nombre d’estimateurs. La validation croisée à 3 plis permet d’évaluer chaque combinaison, puis on affiche les meilleurs paramètres et les précisions sur les ensembles d’entraînement et de test.

In [20]:
# Perform grid search
print("Grid search...")
decision_tree = DecisionTreeClassifier()
model = BaggingClassifier(estimator=decision_tree)

# Grid of values to be tested
params = {
    "estimator__max_depth": [1, 2, 3],
    "estimator__min_samples_leaf": [1, 2, 3],
    "estimator__min_samples_split": [2, 3, 4],
    "n_estimators": [2, 4, 6, 8, 10],
}
print(params)
gridsearch = GridSearchCV(
    model, param_grid=params, cv=3
)  # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'estimator__max_depth': [1, 2, 3], 'estimator__min_samples_leaf': [1, 2, 3], 'estimator__min_samples_split': [2, 3, 4], 'n_estimators': [2, 4, 6, 8, 10]}
...Done.
Best hyperparameters :  {'estimator__max_depth': 1, 'estimator__min_samples_leaf': 1, 'estimator__min_samples_split': 2, 'n_estimators': 2}
Best validation accuracy :  0.5

Accuracy on training set :  0.625
Accuracy on test set :  0.5


Les meilleurs paramètres trouvés pour le Bagging avec arbre de décision sont très simples (`max_depth=1`, `min_samples_leaf=1`, `min_samples_split=2`, `n_estimators=2`). La précision sur le jeu de test reste faible (0.5), et celle sur l’entraînement est seulement légèrement meilleure (0.625).  
Cela indique que le modèle **sous‑apprend** : il est trop simple pour capturer les patterns des données et nécessite soit un arbre plus profond, soit davantage d’estimateurs pour améliorer sa capacité d’apprentissage.



## Boosting


### Scikit-learn's GradientBoosting

#### Boosting with logistic regression as base estimator: forbidden ⛔️


On souhaite tester le **Boosting avec la régression logistique comme modèle de base**, mais cela **n’est pas possible avec `GradientBoostingClassifier`**. En effet, ce classifieur n’accepte **que des arbres de décision comme estimateurs de base**.

Le code montre l’intention de créer un modèle avec `LogisticRegression`, mais il provoque une erreur.  
Cela démontre une **limitation importante** de l’implémentation de Gradient Boosting dans scikit-learn.

In [28]:
# Perform grid search
print("Grid search...")
logistic_regression = LogisticRegression() 
model = GradientBoostingClassifier(logistic_regression) # this will fail: only decision trees are accepted as base estimators in GradientBoosting !

Grid search...


TypeError: GradientBoostingClassifier.__init__() takes 1 positional argument but 2 were given

#### Boosting with decision tree as base estimator

En réalité, seuls les arbres sont utilisés dans les modèles de boosting.
Parce que les arbres de décision sont flexibles, rapides à entraîner et capables de capturer des relations complexes, ce qui les rend idéaux comme modèles faibles dans le boosting.

On effectue une recherche des meilleurs hyperparamètres pour un **GradientBoostingClassifier**.  

La grille teste la profondeur des arbres (`max_depth`), le nombre minimum d’échantillons par feuille et par split, ainsi que le nombre d’estimateurs (`n_estimators`). La validation croisée à 3 plis permet d’évaluer chaque combinaison.  

On affiche ensuite les meilleurs paramètres et les précisions sur les ensembles d’entraînement et de test pour vérifier la performance et la généralisation du modèle.

In [22]:
# Perform grid search
print("Grid search...")
model = GradientBoostingClassifier()

# Grid of values to be tested
params = {
    "max_depth": [
        1,
        2,
        3,
    ],  # no base_estimator_ prefix because these are all arguments of GradientBoostingClassifier
    "min_samples_leaf": [1, 2, 3],
    "min_samples_split": [2, 3, 4],
    "n_estimators": [2, 4, 6, 8, 10],
}
print(params)
gridsearch = GridSearchCV(
    model, param_grid=params, cv=3
)  # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'max_depth': [1, 2, 3], 'min_samples_leaf': [1, 2, 3], 'min_samples_split': [2, 3, 4], 'n_estimators': [2, 4, 6, 8, 10]}
...Done.
Best hyperparameters :  {'max_depth': 1, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 2}
Best validation accuracy :  0.38888888888888884

Accuracy on training set :  0.625
Accuracy on test set :  0.5


Les meilleurs paramètres trouvés sont très simples (`max_depth=1`, `min_samples_leaf=2`, `min_samples_split=2`, `n_estimators=2`). La précision sur le test reste faible (0.5) et celle sur l’entraînement est légèrement meilleure (0.625). Cela montre que le modèle **sous‑apprend** : il est trop limité pour capturer les patterns des données et gagner en performance, il faudrait augmenter la complexité ou le nombre d’estimateurs.

## XGBoost


XGBoost est une bibliothèque différente, mais elle propose une [API compatible scikit-learn ](https://xgboost.readthedocs.io/en/stable/python/python_api.html#module-xgboost.sklearn) qui permet d’entraîner un modèle comme s’il s’agissait d’une classe scikit-learn classique.

On effectue une recherche des meilleurs hyperparamètres pour un **XGBClassifier** via l’API scikit-learn de XGBoost. La grille teste :

* `max_depth` : profondeur des arbres, comme dans scikit-learn,
* `min_child_weight` : similaire à `min_samples_leaf` et `min_samples_split`,
* `n_estimators` : nombre d’arbres dans l’ensemble.

La validation croisée à 3 plis évalue chaque combinaison, puis on affiche les meilleurs paramètres et les précisions sur les ensembles d’entraînement et de test pour vérifier la performance et la généralisation du modèle.

In [25]:
# Perform grid search
print("Grid search...")
xgboost = XGBClassifier()

# Grid of values to be tested
params = {
    "max_depth": [2, 4, 6],  # exactly the same role as in scikit-learn
    "min_child_weight": [
        1,
        2,
        3,
    ],  # effect is more or less similar to min_samples_leaf and min_samples_split
    "n_estimators": [
        2,
        4,
        6,
        8,
    ],  # exactly the same role as in scikit-learn
}
print(params)
gridsearch = GridSearchCV(
    xgboost, param_grid=params, cv=3
)  # cv : the number of folds to be used for CV
gridsearch.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", gridsearch.best_params_)
print("Best validation accuracy : ", gridsearch.best_score_)
print()
print("Accuracy on training set : ", gridsearch.score(X_train, Y_train))
print("Accuracy on test set : ", gridsearch.score(X_test, Y_test))

Grid search...
{'max_depth': [2, 4, 6], 'min_child_weight': [1, 2, 3], 'n_estimators': [2, 4, 6, 8]}
...Done.
Best hyperparameters :  {'max_depth': 2, 'min_child_weight': 1, 'n_estimators': 2}
Best validation accuracy :  0.38888888888888884

Accuracy on training set :  0.5
Accuracy on test set :  0.5


Les meilleurs paramètres trouvés pour XGBoost sont simples (`max_depth=2`, `min_child_weight=1`, `n_estimators=2`).  

La précision sur entraînement et test reste faible (0.5), ce qui montre que le modèle **sous‑apprend** et n’est pas encore adapté pour capturer les patterns des données.



### Récapitulatif des résultats

| Modèle                       | Meilleurs hyperparamètres                                                    | Accuracy train | Accuracy test | Interprétation                                                                                                                         |
| ---------------------------- | ---------------------------------------------------------------------------- | -------------- | ------------- | -------------------------------------------------------------------------------------------------------------------------------------- |
| Bagging + LogisticRegression | `C=0.01`, `n_estimators=10`                                                  | 0.5            | 0.5           | Le modèle est trop simple et ne capture pas les patterns du dataset. Train = Test → stabilité mais sous-apprentissage.                 |
| Bagging + DecisionTree       | `max_depth=1`, `min_samples_leaf=1`, `min_samples_split=2`, `n_estimators=2` | 0.625          | 0.5           | Modèle légèrement plus expressif, mais encore sous-apprenant. Différence entre train et test montre un léger sur-apprentissage limité. |
| GradientBoosting             | `max_depth=1`, `min_samples_leaf=2`, `min_samples_split=2`, `n_estimators=2` | 0.625          | 0.5           | Modèle trop simple, sous-apprentissage. Les scores proches train/test indiquent stabilité malgré la faible performance.                |
| XGBoost                      | `max_depth=2`, `min_child_weight=1`, `n_estimators=2`                        | 0.5            | 0.5           | Sous-apprentissage évident. Train = Test → cohérence, mais dataset trop petit pour exploiter la puissance du modèle.                   |

---

### Interprétation générale

1. **Sous-apprentissage généralisé** : tous les modèles montrent des performances faibles sur test. Les modèles simples (régression logistique, arbres très peu profonds) ne capturent pas la structure des données.

2. **Stabilité relative** : la précision sur train et test est souvent proche, surtout pour Bagging + LogisticRegression et XGBoost. Cela est dû au **très petit dataset (10 lignes)**, qui limite la variance entre train et test.

3. **Limites dues au dataset** : avec si peu d’exemples, même des modèles plus puissants ne peuvent pas améliorer la performance significativement. Les modèles ne surapprennent pas trop, mais ne généralisent pas non plus.

4. **Leçon principale** : pour exploiter pleinement des méthodes comme Bagging ou XGBoost, il faut **plus de données** et/ou **des modèles plus complexes**, ou enrichir les features.


Avec **seulement 10 lignes de données**, aucun modèle complexe ne peut vraiment montrer son avantage. Même XGBoost, qui est puissant sur de gros datasets et pour des patterns complexes, est limité ici. Les résultats faibles ne signifient pas que XGBoost est mauvais ; ils reflètent juste que **le dataset est trop petit pour exploiter sa puissance**.

En d’autres termes :

* Les scores identiques sur train et test pour Bagging LogisticRegression et XGBoost montrent **stabilité**, mais pas performance.
* La véritable force de XGBoost apparaîtrait sur un **dataset plus grand et plus varié**, où il pourrait capturer des interactions complexes et surpasser les modèles simples.
* Ici, le test n’évalue pas la capacité réelle de XGBoost, mais la limite imposée par la taille du dataset.

**Point important** : l’objectif principal ici n’était pas d’obtenir le meilleur score, mais de montrer comment coder et appliquer ces différentes méthodes avec scikit-learn et XGBoost, comment définir une grille d’hyperparamètres, et comment utiliser la validation croisée pour comparer les modèles.

Même si le dataset est limité, cette approche permet de comprendre la mécanique des modèles, ce qui est essentiel avant de passer à des données plus grandes et complexes
